# Seed `PiEvents` — dynamic backfill + live stream

1. **Backfill** the gap: for every tag, generate mostly-normal samples from its **last existing timestamp** up to *now* (any gap size; capped at `BACKFILL_MAX_DAYS`).
2. **Stream** live: emit one fresh point per tag every `STREAM_TICK_SECONDS` for `STREAM_HOURS` hours, so dashboards and 'latest value' answers advance in near-real-time.

Values are drawn around each tag's learned baseline. Seeded rows carry `Source='synthetic-seed'` (`Host` = `SEED-BACKFILL` or `SEED-STREAM`). Set `STREAM_HOURS=0` for backfill only.

**Run all**, then leave the notebook running for the streaming duration.

In [ ]:
# PARAMETERS  (Fabric: this cell is tagged 'parameters')
KUSTO_URI          = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"   # pi-realtime-eventhouse query URI
DATABASE           = "pi-realtime-db"
TABLE              = "PiEvents"
INTERVAL_SECONDS   = 300      # spacing of BACKFILL samples (300 = one every 5 min per tag)
FALLBACK_HOURS     = 24       # if a tag has no prior point, backfill this many hours
BACKFILL_MAX_DAYS  = 7        # safety cap: never start a backfill further back than this
NORMAL_NOISE_FRAC  = 0.15     # noise = fraction of each tag's historical std (small => stable/"normal")
ABNORMAL_FRACTION  = 0.0      # 0 = purely normal; e.g. 0.01 sprinkles mild excursions
DEFAULT_AVG        = 50.0     # fallback baseline for tags with no numeric history
DEFAULT_SD         = 5.0
SEED               = 42
SOURCE_TAG         = "synthetic-seed"

# --- live streaming (after backfill) ---
STREAM_HOURS        = 2.0     # stream continuous values for this many hours (0 = backfill only)
STREAM_TICK_SECONDS = 60      # emit one fresh point per tag every N seconds during streaming


In [ ]:
# Fabric token helpers for the Eventhouse (Kusto): a Spark-connector token + a REST helper for streaming.
import json, time, ssl, urllib.request, urllib.error, datetime as dt

def kusto_token():
    try:
        import notebookutils; cred = notebookutils.credentials
    except Exception:
        from notebookutils import mssparkutils; cred = mssparkutils.credentials
    for aud in (KUSTO_URI, "kusto", "pbi"):
        try:
            t = cred.getToken(aud)
            if t: return t
        except Exception:
            pass
    raise RuntimeError("Could not acquire Kusto token")

_CTX = ssl.create_default_context(); _CTX.check_hostname = False; _CTX.verify_mode = ssl.CERT_NONE

def kusto_mgmt(csl, token):
    body = json.dumps({"db": DATABASE, "csl": csl}).encode()
    req = urllib.request.Request(f"{KUSTO_URI}/v1/rest/mgmt", data=body,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json", "Accept": "application/json"},
        method="POST")
    with urllib.request.urlopen(req, context=_CTX) as r:
        return r.status

TOKEN = kusto_token()
READ_OPTS = {"kustoCluster": KUSTO_URI, "kustoDatabase": DATABASE, "accessToken": TOKEN}
print("Token acquired for", KUSTO_URI)


In [ ]:
# Learn per-tag baseline (real history) and each tag's last timestamp via TWO reads, merged in pandas.
# (A single KQL query that computes percentiles in a let AND joins hits a Kusto plan error, so we split it.)
def kread(kql):
    return (spark.read.format("com.microsoft.kusto.spark.datasource")
            .options(**READ_OPTS).option("kustoQuery", kql).load().toPandas())

base = kread(f"""
PiEvents | where Source != '{SOURCE_TAG}' | extend v = toreal(Value) | where isnotnull(v)
| summarize avg=avg(v), sd=stdev(v), p05=percentile(v,5), p95=percentile(v,95), n=count() by Tag
""")
last = kread("PiEvents | summarize lastTs=max(Ts), Plant=take_any(Plant) by Tag")

import pandas as pd
stats = base.merge(last, on="Tag", how="left")
stats = stats[stats["n"].fillna(0) > 0].reset_index(drop=True)
print("tags with baseline:", len(stats))
stats[["Tag","Plant","lastTs","avg","sd","n"]].head()


In [ ]:
# BACKFILL: for each tag, generate points from its last existing timestamp up to now (capped).
import numpy as np, pandas as pd

rng = np.random.default_rng(SEED)
now = pd.Timestamp.now(tz="UTC")
cap_start = now - pd.Timedelta(days=BACKFILL_MAX_DAYS)
step = pd.Timedelta(seconds=INTERVAL_SECONDS)

def fnum(x, d):
    try: return float(x) if x is not None and not pd.isna(x) else d
    except Exception: return d

frames = []
for r in stats.itertuples(index=False):
    tag  = r.Tag
    last = pd.to_datetime(r.lastTs, utc=True, errors="coerce")
    start = (last + step) if pd.notnull(last) else (now - pd.Timedelta(hours=FALLBACK_HOURS))
    if start < cap_start:
        start = cap_start
    if start > now:
        continue  # tag is already current
    times = pd.date_range(start, now, freq=step)
    if len(times) == 0:
        continue
    base = fnum(r.avg, DEFAULT_AVG)
    sd   = fnum(r.sd, DEFAULT_SD) or (abs(base) * 0.02 + 0.1)
    lo   = fnum(r.p05, base - 2*sd); hi = fnum(r.p95, base + 2*sd)
    if not (hi > lo): lo, hi = base - 3*sd, base + 3*sd
    vals = np.clip(base + rng.normal(0.0, sd * NORMAL_NOISE_FRAC, len(times)), lo, hi)
    if ABNORMAL_FRACTION > 0:
        m = rng.random(len(times)) < ABNORMAL_FRACTION
        vals[m] += rng.normal(0.0, sd * 1.5, int(m.sum()))
    plant = (r.Plant if isinstance(r.Plant, str) and r.Plant else (tag.split(":")[0] if ":" in tag else ""))
    frames.append(pd.DataFrame({
        "Ts": times.tz_convert("UTC").tz_localize(None), "WebId": "", "Tag": tag, "Plant": plant,
        "Value": vals.astype(float), "ValueType": "", "Questionable": False, "Substituted": False,
        "Source": SOURCE_TAG, "Host": "SEED-BACKFILL", "IngestedAt": now.tz_localize(None),
    }))

if frames:
    bdf = pd.concat(frames, ignore_index=True)
else:
    bdf = pd.DataFrame(columns=["Ts","WebId","Tag","Plant","Value","ValueType","Questionable","Substituted","Source","Host","IngestedAt"])
gap_min = 0 if not frames else round((now - pd.to_datetime(stats["lastTs"], utc=True).min()).total_seconds()/60, 1)
print(f"Backfill rows: {len(bdf):,}  (largest gap ~{gap_min} min; window ends {now.isoformat()})")


In [ ]:
# Write the backfill batch via the Kusto Spark connector (skips cleanly if nothing to backfill).
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType, BooleanType, TimestampType)
schema = StructType([
    StructField("Ts", TimestampType(), False), StructField("WebId", StringType(), True),
    StructField("Tag", StringType(), False), StructField("Plant", StringType(), True),
    StructField("Value", DoubleType(), True), StructField("ValueType", StringType(), True),
    StructField("Questionable", BooleanType(), True), StructField("Substituted", BooleanType(), True),
    StructField("Source", StringType(), True), StructField("Host", StringType(), True),
    StructField("IngestedAt", TimestampType(), True),
])
if len(bdf) > 0:
    sdf = spark.createDataFrame(bdf, schema=schema)
    try:
        (sdf.write.format("com.microsoft.kusto.spark.datasource")
            .option("kustoCluster", KUSTO_URI).option("kustoDatabase", DATABASE)
            .option("kustoTable", TABLE).option("accessToken", TOKEN)
            .option("tableCreateOptions", "FailIfNotExist").option("writeMode", "Queued")
            .mode("Append").save())
    except Exception as e:
        print("Connector raised after submitting ingestion (validating):", str(e)[:160])
    print("Backfill submitted:", len(bdf), "rows")
else:
    print("No backfill needed - all tags already current.")


In [ ]:
# STREAM: emit one fresh point per tag every STREAM_TICK_SECONDS for STREAM_HOURS (near-real-time).
import numpy as np, pandas as pd, datetime as dt, time

def fnum(x, d):
    try: return float(x) if x is not None and not pd.isna(x) else d
    except Exception: return d

# Pre-compute per-tag baseline arrays for fast tick generation.
tags   = stats["Tag"].tolist()
plants = [ (p if isinstance(p, str) and p else (t.split(":")[0] if ":" in t else "")) for t, p in zip(tags, stats["Plant"].tolist()) ]
bases  = np.array([fnum(v, DEFAULT_AVG) for v in stats["avg"].tolist()])
sds    = np.array([fnum(s, DEFAULT_SD) or 1.0 for s in stats["sd"].tolist()])
los    = np.array([fnum(v, b - 2*s) for v, b, s in zip(stats["p05"].tolist(), bases, sds)])
his    = np.array([fnum(v, b + 2*s) for v, b, s in zip(stats["p95"].tolist(), bases, sds)])
rng2   = np.random.default_rng(SEED + 1)

def esc(v):
    return "" if v is None else str(v)

def make_csv(ts):
    vals = np.clip(bases + rng2.normal(0.0, sds * NORMAL_NOISE_FRAC), los, his)
    tsz = ts.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
    lines = []
    for i, tag in enumerate(tags):
        # column order: Ts,WebId,Tag,Plant,Value,ValueType,Questionable,Substituted,Source,Host,IngestedAt
        lines.append(f"{tsz},,{tag},{plants[i]},{vals[i]:.6f},,false,false,{SOURCE_TAG},SEED-STREAM,{tsz}")
    return "\n".join(lines)

if STREAM_HOURS and STREAM_HOURS > 0:
    end = dt.datetime.now(dt.timezone.utc) + dt.timedelta(hours=float(STREAM_HOURS))
    tick = 0; ok = 0; tok = TOKEN; last_tok = time.time()
    print(f"Streaming until {end.isoformat()} (tick every {STREAM_TICK_SECONDS}s, {len(tags)} tags/tick)...")
    while dt.datetime.now(dt.timezone.utc) < end:
        t0 = time.time()
        now_ts = dt.datetime.now(dt.timezone.utc)
        try:
            if time.time() - last_tok > 1500:   # refresh token every ~25 min
                tok = kusto_token(); last_tok = time.time()
            csl = ".ingest inline into table ['" + TABLE + "'] <|\n" + make_csv(now_ts)
            kusto_mgmt(csl, tok)
            ok += 1
        except Exception as e:
            print(f"  tick {tick} ingest error:", str(e)[:150])
        tick += 1
        if tick % 5 == 0:
            print(f"  {now_ts.strftime('%H:%M:%S')}Z  ticks={tick} ok={ok}")
        sleep = STREAM_TICK_SECONDS - (time.time() - t0)
        if sleep > 0 and dt.datetime.now(dt.timezone.utc) < end:
            time.sleep(sleep)
    print(f"Streaming complete: {tick} ticks, {ok} succeeded, ~{ok*len(tags):,} points emitted.")
else:
    print("Streaming disabled (STREAM_HOURS=0).")


In [ ]:
# Quick check: newest synthetic point (should be ~now) and total synthetic rows.
q = f"PiEvents | where Source == '{SOURCE_TAG}' | summarize rows=count(), tags=dcount(Tag), newest=max(Ts)"
try:
    (spark.read.format("com.microsoft.kusto.spark.datasource")
        .options(**READ_OPTS).option("kustoQuery", q).load().show(truncate=False))
except Exception as e:
    print("verify skipped:", e)
